# Understanding the Data
After unpacking the 2011-2023 Gwinnett zip file, I saw that there were actual a number of the excel spreadsheets which did contain sales information.

The format of this information is a little interesting, because there's several columns that are used to identify:
- LRSNum
- PIN
- LOCADDR
- LocCity
- LocZip
- LEGALAC
- PCDESC
- ZONEDESC

Then some attributes that are used to identify a specific owner, in this context a grantee.
- OWNER1
- OWNER2
- MAILADDR
- MAILCITY
- MAILSTAT
- Sale Date (Broken into SALE1D, SALE2D, SALE3D for the last transcations)
- Sale Amount (Broken into SALE1AMT, SALE2AMT, SALE3AMT)
- Grantor (Broken into GRANTOR1, GRANTOR2, GRANTOR3)
- Document Reference (Broken into DOC1REF, DOC2REF, DOC3REF) - I'm assuming something like the page and book numbers in other counties

The creation of the sales records is interesting because OWNER1 is the GRANTEE of GRANTOR1, GRANTOR1 is the GRANTEE of GRANTOR2, etc.  
So only some of the records will have OWNER1, OWNER2, MAILADDR, MAILCITY, MAILSTAT, specifically those records corresponding to the final sale of the contemporaneous owner (at time of recording).

**Road Map**  
1. Pick out a singular tax assessment document per year. There seems to be a lot of duplicate versions, with largely the same information. I'm just picking out the sheet that has the most rows / the latest sales date, indicating it's the most updated.
2. Check to ensure that the important columns are present across all of the assessment documents.
3. Duplicate the relevant property-wise columns for each of the sales transactions, to produce a unique row for each sale.
4. Merge all the years sales data.
5. Sort by those rows which have non-empty MAILADDR columns.
6. Deduplicate, taking the first value, ensuring that we take the most up to date sales transactions (those with MAILADDR), if possible

**Selected Tax Assessment Documents**  
Current Ownership_2011 Digest Assessed Values.xlsx  
2012 Gwinnett Digest TAFull_Ownership_CD7.xlsx  
2013 Tax Digest Ownership_CD7.xlsx  
2014 Property Ownership CD7.xlsx  
2015 Property Ownership CD7.xlsx  
2016 Property Ownership CD7.xlsx  
2017 Property Ownership CD7.xlsx  
2018 Property Ownership  CD7.xlsx  
2019 Property Ownership CD7.xlsx  
2020 Property Ownership CD7.xlsx  
2021 Property Ownership CD7.xlsx  
2022 Property Ownership CD7.xlsx  
2023 Property Ownership CD7.xlsx  

In [1]:
import pandas as pd
import os
from datetime import datetime

DATA_PATH = "../../data/gwinnett"
OUT_PATH = "../../data/gwinnett/out"

I have renamed all of hte files to follow the 20{XX} Property Ownership CD7.xlsx format, for convenience.

In [5]:
year_dfs = []

for file_p in os.listdir(DATA_PATH):
    if file_p.endswith(".xlsx"):
        year = int(file_p.split(" ")[0])
        df = pd.read_excel(os.path.join(DATA_PATH, file_p))

        year_dfs.append((year, df))

In [6]:
year_dfs.sort(key = lambda x : x[0])

In [121]:
# 2013 has an abnormal page structure
year_dfs[2] = (2013, pd.read_excel(os.path.join(DATA_PATH, "2013 Property Ownership CD7.xlsx"), sheet_name="real_master_0001"))

In [122]:
for year, df in year_dfs:
    print(year)
    print(df.columns)

2011
Index(['LRSNum', 'PIN', 'Public_NeiNum', 'LOCADDR', 'LocCity', 'LocState',
       'LocZip', 'OWNER1', 'OWNER2', 'MAILADDR', 'MAILCITY', 'MAILSTAT',
       'MAILZIP', 'LEGALAC', 'PCDESC', 'ZONEDESC', 'EXEMPT1', 'EXEMPT1D',
       'ASSMNT1D', 'LANDVAL1', 'DWLGVAL1', 'OTHVAL1', 'TOTVAL1', 'TAXLAND1',
       'TAXDWLG1', 'TAXOTH1', 'TAXTOT1', 'SALE1D', 'SALE2D', 'SALE3D',
       'SALE1AMT', 'SALE2AMT', 'SALE3AMT', 'GRANTOR1', 'GRANTOR2', 'GRANTOR3',
       'DOC1REF', 'DOC2REF', 'DOC3REF', 'YEAR_RECORDED'],
      dtype='object')
2012
Index(['LRSNum', 'PIN', 'Public_NeiNum', 'LOCADDR', 'LocCity', 'LocState',
       'LocZip', 'OWNER1', 'OWNER2', 'MAILADDR', 'MAILCITY', 'MAILSTAT',
       'MAILZIP', 'LEGALAC', 'PCDESC', 'ZONEDESC', 'EXEMPT1', 'EXEMPT1D',
       'ASSMNT1D', 'LANDVAL1', 'DWLGVAL1', 'OTHVAL1', 'TOTVAL1', 'TAXLAND1',
       'TAXDWLG1', 'TAXOTH1', 'TAXTOT1', 'SALE1D', 'SALE2D', 'SALE3D',
       'SALE1AMT', 'SALE2AMT', 'SALE3AMT', 'GRANTOR1', 'GRANTOR2', 'GRANTOR3',
       'DOC1

In [123]:
for year, df in year_dfs:
    print(f"Not in 2023, in {year}")
    print(set(df.columns).difference(set(year_dfs[-1][1].columns)))
    print(f"Not in {year}, in 2023")
    print(set(year_dfs[-1][1].columns).difference(set(df.columns)))
    print("\n")

Not in 2023, in 2011
{'LocCity', 'MAILCITY', 'GRANTOR1', 'GRANTOR2', 'LOCADDR', 'LocState', 'GRANTOR3', 'EXEMPT1D', 'OWNER1', 'EXEMPT1', 'OWNER2', 'MAILADDR', 'MAILSTAT', 'MAILZIP', 'LocZip'}
Not in 2011, in 2023
{'OWNERNAME1', 'GRANTORNAME2', 'PROPCLAS', 'DISTNUM_DESC', 'PROPERTYState', 'OWNERCITY', 'OWNERZIP', 'OWNERADDRESS1', 'LEGAL1', 'OWNERSTATE', 'DISTNUM', 'PROPCLAS_DESC', 'OWNERNAME2', 'GRANTORNAME3', 'PROPERTYZip', 'PROPERTYCity', 'PROPERTYSTREET', 'GRANTORNAME1'}


Not in 2023, in 2012
{'LocCity', 'MAILCITY', 'GRANTOR1', 'GRANTOR2', 'LOCADDR', 'LocState', 'GRANTOR3', 'EXEMPT1D', 'OWNER1', 'EXEMPT1', 'OWNER2', 'MAILADDR', 'MAILSTAT', 'MAILZIP', 'LocZip'}
Not in 2012, in 2023
{'OWNERNAME1', 'GRANTORNAME2', 'PROPCLAS', 'DISTNUM_DESC', 'PROPERTYState', 'OWNERCITY', 'OWNERZIP', 'OWNERADDRESS1', 'LEGAL1', 'OWNERSTATE', 'DISTNUM', 'PROPCLAS_DESC', 'OWNERNAME2', 'GRANTORNAME3', 'PROPERTYZip', 'PROPERTYCity', 'PROPERTYSTREET', 'GRANTORNAME1'}


Not in 2023, in 2013
{'LocCity', 'MAILCI

It looks like 2011 and 2012 have their own format, 2013 has its own format, and the rest of the years are the same. So I will be parsing them using the columns of the 2023 dataset.

In [ ]:
property_cols = ['LRSNum', 'PIN', 'Public_NeiNum', 'LOCADDR', 'LocCity', 'LocState',
       'LocZip', 'LEGALAC', 'PCDESC', 'ZONEDESC']

owner_cols = ['OWNER1', 'OWNER2', 'MAILADDR', 'MAILCITY', 'MAILSTAT',
       'MAILZIP']

drop_cols = ['EXEMPT1', 'EXEMPT1D',
       'ASSMNT1D', 'LANDVAL1', 'DWLGVAL1', 'OTHVAL1', 'TOTVAL1', 'TAXLAND1',
       'TAXDWLG1', 'TAXOTH1', 'TAXTOT1']

sale_cols = ['SALE1D', 'SALE2D', 'SALE3D',
       'SALE1AMT', 'SALE2AMT', 'SALE3AMT', 'GRANTOR1', 'GRANTOR2', 'GRANTOR3',
       'DOC1REF', 'DOC2REF', 'DOC3REF']

sales_2011_2013 = []
for i in range(3):
    year, year_df = year_dfs[i]
    year_df['SALE1D'] = pd.to_datetime(year_df['SALE1D'], format="%m/%d/%Y", errors="coerce")
    year_df['SALE2D'] = pd.to_datetime(year_df['SALE2D'], format="%m/%d/%Y", errors="coerce")
    year_df['SALE3D'] = pd.to_datetime(year_df['SALE3D'], format="%m/%d/%Y", errors="coerce")
    year_df['YEAR_RECORDED'] = year
    grantor3_sales = year_df.loc[~year_df['SALE3D'].isna(), :].copy()
    grantor2_sales = year_df.loc[~year_df['SALE2D'].isna(), :].copy()
    grantor1_sales = year_df.loc[~year_df['SALE1D'].isna(), :].copy()
    
    grantor3_sales['GRANTOR'] = grantor3_sales['GRANTOR3']
    grantor3_sales['GRANTEE'] = grantor3_sales['GRANTOR2']
    grantor3_sales['SALEDT'] = grantor3_sales['SALE3D']
    grantor3_sales['SALEAMT'] = grantor3_sales['SALE3AMT']
    grantor3_sales['DOCREF'] = grantor3_sales['DOC3REF']
    grantor3_sales.loc[:, owner_cols] = pd.NA
    grantor3_sales = grantor3_sales.drop(columns=drop_cols + sale_cols)

    grantor2_sales['GRANTOR'] = grantor2_sales['GRANTOR2']
    grantor2_sales['GRANTEE'] = grantor2_sales['GRANTOR1']
    grantor2_sales['SALEDT'] = grantor2_sales['SALE2D']
    grantor2_sales['SALEAMT'] = grantor2_sales['SALE2AMT']
    grantor2_sales['DOCREF'] = grantor2_sales['DOC2REF']
    grantor2_sales.loc[:, owner_cols] = pd.NA
    grantor2_sales = grantor2_sales.drop(columns=drop_cols + sale_cols)

    grantor1_sales['GRANTOR'] = grantor1_sales['GRANTOR1']
    grantor1_sales['GRANTEE'] = grantor1_sales['OWNER1']
    grantor1_sales['SALEDT'] = grantor1_sales['SALE1D']
    grantor1_sales['SALEAMT'] = grantor1_sales['SALE1AMT']
    grantor1_sales['DOCREF'] = grantor1_sales['DOC1REF']
    # Notably, no clearing of the owner1 columns this time because they actually exist
    grantor1_sales = grantor1_sales.drop(columns=drop_cols + sale_cols)

    sales_2011_2013.append(grantor1_sales)
    sales_2011_2013.append(grantor2_sales)
    sales_2011_2013.append(grantor3_sales)

sales_2011_2013_df = pd.concat(sales_2011_2013)

In [127]:
# Normalize with the other dataframes
sales_2011_2013_df = sales_2011_2013_df.rename(columns={'LOCADDR': 'PROPERTYSTREET', 'LocCity': 'PROPERTYCity',
                                                        'LocState': 'PROPERTYState', 'LocZip': 'PROPERTYZip',
                                                        'OWNER1': 'OWNERNAME1', 'OWNER2': 'OWNERNAME2',
                                                        'MAILADDR': 'OWNERADDRESS1', 'MAILCITY': "OWNERCITY",
                                                        "MAILSTAT": "OWNERSTATE", "MAILZIP": "OWNERZIP"})

In [128]:
property_cols = ['LRSNum', 'PIN', 'Public_NeiNum', 'PROPERTYSTREET', 'PROPERTYCity',
       'PROPERTYState', 'PROPERTYZip', 'LEGALAC', 'LEGAL1', 'PCDESC', 'ZONEDESC']

owner_cols = ['OWNERNAME1', 'OWNERNAME2',
       'OWNERADDRESS1', 'OWNERCITY', 'OWNERSTATE', 'OWNERZIP']

drop_cols = ['ASSMNT1D', 'LANDVAL1', 'DWLGVAL1', 'OTHVAL1', 'TOTVAL1', 'TAXLAND1', 'TAXDWLG1',
       'TAXOTH1', 'TAXTOT1', 'LEGAL1', ]

sale_cols = ['SALE1D', 'SALE2D', 'SALE3D', 'SALE1AMT',
       'SALE2AMT', 'SALE3AMT', 'GRANTORNAME1', 'GRANTORNAME2', 'GRANTORNAME3',
       'DOC1REF', 'DOC2REF', 'DOC3REF']

sales_2014_2023 = []
for i in range(3, len(year_dfs)):
    year, year_df = year_dfs[i]
    year_df['SALE1D'] = pd.to_datetime(year_df['SALE1D'], format="%m/%d/%Y", errors="coerce")
    year_df['SALE2D'] = pd.to_datetime(year_df['SALE2D'], format="%m/%d/%Y", errors="coerce")
    year_df['SALE3D'] = pd.to_datetime(year_df['SALE3D'], format="%m/%d/%Y", errors="coerce")
    year_df['YEAR_RECORDED'] = year
    grantor3_sales = year_df.loc[~year_df['SALE3D'].isna(), :].copy()
    grantor2_sales = year_df.loc[~year_df['SALE2D'].isna(), :].copy()
    grantor1_sales = year_df.loc[~year_df['SALE1D'].isna(), :].copy()
    
    grantor3_sales['GRANTOR'] = grantor3_sales['GRANTORNAME3']
    grantor3_sales['GRANTEE'] = grantor3_sales['GRANTORNAME2']
    grantor3_sales['SALEDT'] = grantor3_sales['SALE3D']
    grantor3_sales['SALEAMT'] = grantor3_sales['SALE3AMT']
    grantor3_sales['DOCREF'] = grantor3_sales['DOC3REF']
    grantor3_sales.loc[:, owner_cols] = pd.NA
    grantor3_sales = grantor3_sales.drop(columns=drop_cols + sale_cols)

    grantor2_sales['GRANTOR'] = grantor2_sales['GRANTORNAME2']
    grantor2_sales['GRANTEE'] = grantor2_sales['GRANTORNAME1']
    grantor2_sales['SALEDT'] = grantor2_sales['SALE2D']
    grantor2_sales['SALEAMT'] = grantor2_sales['SALE2AMT']
    grantor2_sales['DOCREF'] = grantor2_sales['DOC2REF']
    grantor2_sales.loc[:, owner_cols] = pd.NA
    grantor2_sales = grantor2_sales.drop(columns=drop_cols + sale_cols)

    grantor1_sales['GRANTOR'] = grantor1_sales['GRANTORNAME1']
    grantor1_sales['GRANTEE'] = grantor1_sales['OWNERNAME1']
    grantor1_sales['SALEDT'] = grantor1_sales['SALE1D']
    grantor1_sales['SALEAMT'] = grantor1_sales['SALE1AMT']
    grantor1_sales['DOCREF'] = grantor1_sales['DOC1REF']
    # Notably, no clearing of the owner1 columns this time because they actually exist
    grantor1_sales = grantor1_sales.drop(columns=drop_cols + sale_cols)

    sales_2014_2023.append(grantor1_sales)
    sales_2014_2023.append(grantor2_sales)
    sales_2014_2023.append(grantor3_sales)

sales_2014_2023_df = pd.concat(sales_2014_2023)
sales_2014_2023_df = sales_2014_2023_df.drop(columns=['EXEMPT1', 'EXEMPT1D'])

In [35]:
total_sales_digest = pd.concat([sales_2011_2013_df, sales_2014_2023_df])

In [ ]:
total_sales_digest['HASOWNER'] = ~total_sales_digest['OWNERNAME1'].isna()
total_sales_digest['PIN'] = total_sales_digest['PIN'].str.strip()

In [70]:
total_sales_digest = total_sales_digest.sort_values(by="HASOWNER", ascending=False)
total_sales_digest['SALE_YR'] = pd.to_datetime(total_sales_digest['SALEDT']).dt.year

I can't actually distinguish for sure which one is the "ParcelID" equivalent for Gwinnett county. Both LRSNum and PIN seem to be unique. I just choose PIN

In [ ]:
original_rows = total_sales_digest.shape[0]

total_sales_digest_dedup = total_sales_digest.drop_duplicates(subset=['PIN', 'SALEDT'])
new_rows = total_sales_digest_dedup.shape[0]

print(f"Rows Removed: {original_rows - new_rows}, {new_rows} Remaining")

Rows Removed: 7552224, 936953 Remaining


This passes the sniff test, since the total number of 2023 rows was around 300000, so assuming that there were some properties which were sold more than 3 times, wherein the 2012 file contributed different sales records than 2023 for example, then this checks out.

In [72]:
total_sales_digest_dedup = total_sales_digest_dedup.sort_values(by="SALEDT", ascending=True)

In [73]:
total_sales_digest_dedup.to_csv(os.path.join(OUT_PATH, "GWINNET_SALES_RAW.csv"), index=False)

# Merging with Tax Digest
The sales digest in this scenario is actually derived from the tax digest, so it is a little redundant, but I guess it's a reverse mapping if you need that.

In [79]:
TAX_DIGEST_PATH = "/Users/tpeng/Library/CloudStorage/OneDrive-GeorgiaInstituteofTechnology/Housing and Urban Policy (HUP) Lab - Documents/Data Engineering/Output/Most up-to date/Gwinnett/3-ownership_keys/gwinnett_digest_full_final.csv"
SALES_DIGEST_PATH = os.path.join(OUT_PATH, "GWINNET_SALES_RAW.csv")

print(f"Correct Tax Digest Path {os.path.exists(TAX_DIGEST_PATH)}")
print(f"Correct Sales Digest Path {os.path.exists(SALES_DIGEST_PATH)}")

Correct Tax Digest Path True
Correct Sales Digest Path True


In [55]:
tax_digest = pd.read_csv(TAX_DIGEST_PATH, dtype={"property_zip": object, "legal1": object, "distnum": object, 
                                                 "distnum_desc": object, "propclas_desc": object,
                                                 "mod_ownerzip": object}, 
                                                 parse_dates=["assmnt1d", "sale1d", "sale2d", "sale3d"])

There are duplicate year - pin, year - lrs_num pairs in the gwinnett tax digest.

In [69]:
# Determine which of LRSNum and Pin are the unique identifier
print(f"lrs_num unique: {~tax_digest.duplicated(subset=['lrs_num', 'tax_year']).any()}")
conflicts = tax_digest[tax_digest.duplicated(subset=['lrs_num', 'tax_year'], keep=False)]
print(conflicts)
print("\n")

print(f"pin unique: {~tax_digest.duplicated(subset=['pin', 'tax_year']).any()}")
conflicts = tax_digest[tax_digest.duplicated(subset=['pin', 'tax_year'], keep=False)]
print(conflicts)
print("\n")

lrs_num unique: False
         tax_year   lrs_num        pin  public_nei_num         propertystreet  \
265320       2011  33318799  R7309 226               0  5154 BELMORE MANOR CT   
265321       2011  33318799  R7309 226               0  5154 BELMORE MANOR CT   
265322       2011  33318799  R7309 226               0  5154 BELMORE MANOR CT   
265323       2011  33318799  R7309 226               0  5154 BELMORE MANOR CT   
1047106      2014  33335279  R7114 304               0     1695 GLENHAVEN WAY   
...           ...       ...        ...             ...                    ...   
3382233      2022  33419810  R7287 417            7541           49 OLIVE WAY   
3382234      2022  33419811  R7287 418               0           69 OLIVE WAY   
3382235      2022  33419811  R7287 418               0           69 OLIVE WAY   
3382236      2022  33419811  R7287 418            7541           69 OLIVE WAY   
3382237      2022  33419811  R7287 418            7541           69 OLIVE WAY   

     

In [80]:
sales_digest = pd.read_csv(SALES_DIGEST_PATH)

/var/folders/bb/g7vlcgfn14ld8_w41331g0dw0000gn/T/ipykernel_79867/682345646.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  sales_digest = pd.read_csv(SALES_DIGEST_PATH)


In [82]:
merge_cols = ['tax_year', 'lrs_num', 'pin', 'exempt1', 'exempt1d', 'assmnt1d',
       'landval1', 'dwlgval1', 'othval1', 'totval1', 'taxland1', 'taxdwlg1',
       'taxoth1', 'taxtot1', 'mod_own_adrstr', 'mod_unitno',
       'mod_own_adrsuf2', 'mod_own_adrsuf', 'mod_ownerzip', 'owner_addr',
       'own_corp_flag', 'rental_flag', 'mod_owneraddress1',
       'mod_owneraddress1_B']

tax_subset = tax_digest[merge_cols]
sales_subset = sales_digest[sales_digest['SALE_YR'] >= 2011]
sales_tax_merged = pd.merge(sales_subset, tax_subset, how="left", left_on=["PIN", "SALE_YR"], right_on=["pin", "tax_year"])

total_rows = sales_subset.shape[0]
merged_rows = sales_tax_merged['pin'].notna().sum()

print(f"{merged_rows} out of {total_rows} matched: {merged_rows / total_rows * 100}%")

390803 out of 398852 matched: 97.98195822009166%


In [99]:
missing_pins = sales_tax_merged[sales_tax_merged['pin'].isna()].head(10)["PIN"].unique()
print(missing_pins)
print(f"Total Missing: {len(missing_pins)}")

['R5134 253' 'R7232 407' 'R7138 448' 'R7285 186' 'R7285 188' 'R7082 301'
 'R7082 302' 'R7011 134' 'R6129 011']
Total Missing: 9


In [130]:
tax_digest[tax_digest['pin'].isin(missing_pins)].sort_values(by="pin")

,tax_year,lrs_num,pin,public_nei_num,propertystreet,property_city,property_state,property_zip,ownername1,ownername2,...,mod_own_adrstr,mod_unitno,mod_own_adrsuf2,mod_own_adrsuf,mod_ownerzip,owner_addr,own_corp_flag,rental_flag,mod_owneraddress1,mod_owneraddress1_B
3470933,2023,33271689,R5134 253,5639,2431 ALEXANDER TOP PLACE,GRAYSON,NaN,NaN,NORRIS-CONSTABLE GLENDA-MARIE P,CONSTABLE ALBERT A,...,PO BOX 1943,NaN,NaN,NaN,30096,PO BOX 1943 30096,0,1,PO BOX 1943,PO BOX 1943
3170690,2022,33271689,R5134 253,5639,2431 ALEXANDER TOP PLACE,GRAYSON,NaN,NaN,NORRIS-CONSTABLE GLENDA-MARIE P,CONSTABLE ALBERT A,...,PO BOX 1943,NaN,NaN,NaN,30096,PO BOX 1943 30096,0,1,PO BOX 1943,PO BOX 1943
2873885,2021,33271689,R5134 253,5639,2431 ALEXANDER TOP PLACE,GRAYSON,NaN,NaN,BARRON HOMES OF GEORGIA INC,NaN,...,113 N BAILEY,NaN,NaN,LN,20132,113 N BAILEY 20132,1,1,113 N BAILEY LN,113 N BAILEY LN
2007501,2018,33271689,R5134 253,5014,2431 ALEXANDER TOP PLACE,GRAYSON,NaN,NaN,BARRON HOMES OF GEORGIA INC,NaN,...,113 N BAILEY,NaN,NaN,LN,20132,113 N BAILEY 20132,1,1,113 N BAILEY LN,113 N BAILEY LN
2582701,2020,33271689,R5134 253,5014,2431 ALEXANDER TOP PLACE,GRAYSON,NaN,NaN,BARRON HOMES OF GEORGIA INC,NaN,...,113 N BAILEY,NaN,NaN,LN,20132,113 N BAILEY 20132,1,1,113 N BAILEY LN,113 N BAILEY LN
2293827,2019,33271689,R5134 253,5014,2431 ALEXANDER TOP PLACE,GRAYSON,NaN,NaN,BARRON HOMES OF GEORGIA INC,NaN,...,113 N BAILEY,NaN,NaN,LN,20132,113 N BAILEY 20132,1,1,113 N BAILEY LN,113 N BAILEY LN
2652739,2020,829579,R6129 011,6010,SIR GREGORY MANOR,LAWRENCEVILLE,NaN,30044,TAYLOR MORRISON OF GEORGIA LLC,NaN,...,1959 CHARCOAL IVES,NaN,NaN,RD,30045,1959 CHARCOAL IVES 30045,1,1,1959 CHARCOAL IVES RD,1959 CHARCOAL IVES RD
2363247,2019,829579,R6129 011,6010,SIR GREGORY MANOR,LAWRENCEVILLE,NaN,30044,TAYLOR MORRISON OF GEORGIA LLC,NaN,...,4400 N POINT,295,NaN,PKWY,30022,4400 N POINT 295 30022,1,1,4400 N POINT PKWY,4400 N POINT PKWY
132081,2011,829579,R6129 011,6010,BETHANY CHURCH RD,LAWRENCEVILLE,NaN,30044,GROVES EDWARD A,NaN,...,301 N RIVER,NaN,SW,DR,30047,301 N RIVER 30047,0,1,301 N RIVER DR SW,301 N RIVER DR
2076330,2018,829579,R6129 011,6010,SIR GREGORY MANOR,LAWRENCEVILLE,NaN,30044,TAYLOR MORRISON OF GEORGIA LLC,NaN,...,4400 N POINT,295,NaN,PKWY,30022,4400 N POINT 295 30022,1,1,4400 N POINT PKWY,4400 N POINT PKWY


In [ ]:
sales_digest[sales_digest['PIN'] == "R7232 407"]

,LRSNum,PIN,Public_NeiNum,PROPERTYSTREET,PROPERTYCity,PROPERTYState,PROPERTYZip,OWNERNAME1,OWNERNAME2,OWNERADDRESS1,...,GRANTEE,SALEDT,SALEAMT,DOCREF,DISTNUM,DISTNUM_DESC,PROPCLAS,PROPCLAS_DESC,HASOWNER,SALE_YR
541556,33353538,R7232 407,7895,775 LAURA JEAN CT,BUFORD,NaN,30518,NaN,NaN,NaN,...,WOODWARD MILL DEVELOPMENT LLC,2011-02-23,435824.0,55728 291,NaN,NaN,NaN,NaN,False,2011
754973,33353538,R7232 407,7895,775 LAURA JEAN CT,BUFORD,NaN,30518,SORRELLS JERRY A II,QUIJANO-SORRELLS VIVIANA,775 LAURA JEAN CT,...,SORRELLS JERRY A II,2017-12-12,7218303.0,55589 742,NaN,NaN,NaN,NaN,True,2017
761116,33353538,R7232 407,7895,775 LAURA JEAN CT,BUFORD,,30518,SORRELLS JERRY A II,QUIJANO-SORRELLS VIVIANA,775 LAURA JEAN CT,...,SORRELLS JERRY A II,2018-02-23,435824.0,55728 291,01,COUNTY Unincorporated,101.0,R,True,2018


: 

In [ ]:
sales_digest[sales_digest['pin'].isin(missing_pins)]

In [101]:
sales_tax_merged[sales_tax_merged['pin'].isna()]['SALE_YR'].value_counts().sort_index(ascending=True)

SALE_YR
2011       4
2012      52
2013      89
2014     229
2015     404
2016     714
2017    1085
2018     611
2019     889
2020    2160
2021    1050
2022     816
2102       1
2111       1
Name: count, dtype: int64

In [129]:
sales_tax_merged.to_csv(os.path.join(OUT_PATH, "GWINNET_SALES_TAX_DIGEST_FINAL.csv"), index=False)